### 검색도구: Tavily Search

LangChain에는 Tavily 검색 엔진을 도구로 쉽게 사용할 수 있는 내장 도구가 있습니다.

Tavily Search 를 사용하기 위해서는 API KEY를 발급 받아야 합니다.

- [Tavily Search API 발급받기](https://app.tavily.com/sign-in)

발급 받은 API KEY 를 다음과 같이 환경변수에 등록 합니다.

아래 코드의 주석을 풀고 발급받은 API KEY 를 설정합니다.


In [1]:
import os

# TAVILY API KEY를 기입합니다.
os.environ["TAVILY_API_KEY"] = "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"

# 디버깅을 위한 프로젝트명을 기입합니다.
os.environ["LANGCHAIN_PROJECT"] = "AGENT TUTORIAL"
os.environ["LANGCHAIN_TRACING_V2"] = "False"

In [2]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

False

In [3]:
# TavilySearchResults 클래스를 langchain_community.tools.tavily_search 모듈에서 가져옵니다.
from langchain_community.tools.tavily_search import TavilySearchResults

# TavilySearchResults 클래스의 인스턴스를 생성합니다
# k=5은 검색 결과를 5개까지 가져오겠다는 의미입니다
search = TavilySearchResults(k=3)

### Agent 가 사용할 도구 목록 정의


In [4]:
# tools 리스트에 search 도구를 추가합니다.
tools = [search]

## Agent with smaller model

- reference: https://github.com/langchain-ai/langchain/blob/master/templates/neo4j-semantic-ollama/neo4j_semantic_ollama/agent.py


In [5]:
from langchain.agents import AgentExecutor
from langchain.agents.format_scratchpad import format_log_to_messages
from langchain.agents.output_parsers import (
    ReActJsonSingleInputOutputParser,
)
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.pydantic_v1 import BaseModel, Field
from langchain.tools.render import render_text_description_and_args
from langchain_core.messages import AIMessage, HumanMessage

from langchain.pydantic_v1 import BaseModel, Field


local_llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="cognitivecomputations/dolphin-2.9-llama3-8b-gguf",
    temperature=0,
)

tools = [search]

llm_with_tools = local_llm.bind_tools(tools)

In [6]:
from typing import Tuple, List

chat_model_with_stop = local_llm.bind(stop=["Observation", "\nObservation", "\n관측"])

# Inspiration taken from hub.pull("hwchase17/react-json")
system_message = f"""Answer the following questions as best you can.
You can answer directly if the user is greeting you or similar.
Otherise, you have access to the following tools:

{render_text_description_and_args(tools).replace('{', '{{').replace('}', '}}')}

The way you use the tools is by specifying a json blob.
Specifically, this json should have a `action` key (with the name of the tool to use)
and a `action_input` key (with the input to the tool going here).
The only values that should be in the "action" field are: {[t.name for t in tools]}
The $JSON_BLOB should only contain a SINGLE action, 
do NOT return a list of multiple actions.
Here is an example of a valid $JSON_BLOB:
```
{{{{
    "action": $TOOL_NAME,
    "action_input": $INPUT
}}}}
```
The $JSON_BLOB must always be enclosed with triple backticks!

ALWAYS use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action:```
$JSON_BLOB
```
Observation: the result of the action... 
(this Thought/Action/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin! Reminder to always use the exact characters `Final Answer` when responding.'
"""

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "user",
            system_message,
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)


def _format_chat_history(chat_history: List[Tuple[str, str]]):
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

In [7]:
agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_log_to_messages(x["intermediate_steps"]),
        "chat_history": lambda x: (
            _format_chat_history(x["chat_history"]) if x.get(
                "chat_history") else []
        ),
    }
    | prompt
    | chat_model_with_stop
    | ReActJsonSingleInputOutputParser()
)


# Add typing for input
class AgentInput(BaseModel):
    input: str
    chat_history: List[Tuple[str, str]] = Field(
        ..., extra={"widget": {"type": "chat", "input": "input", "output": "output"}}
    )


agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
).with_types(input_type=AgentInput)

In [8]:
# 검색 결과를 요청 후 질문에 대한 답변을 출력합니다.
response = agent_executor.invoke(
    {
        "input": "고척 신선설농탕 전화번호를 웹 검색하여 결과를 알려주세요.",
        "chat_history": [],
    }
)
print(f'답변: {response["output"]}')



> Entering new AgentExecutor chain...
Question: 고척 신선설농탕 전화번호를 웹 검색하여 결과를 알려주세요.
Thought: I will search for the phone number of Gocek Shinseolnongtang using a web search engine.
Action: ```{
     "action": "tavily_search_results_json",
     "action_input": {
         "query": "고척 신선설농탕 전화번호"
     }
}```

[{'url': 'https://www.diningcode.com/profile.php?rid=tBA1JWbJxcdM', 'content': 'chazer 4월 24일. 구로구 고척동 맛집 신선설농탕 고척점 가격메뉴주차운영시간리뷰 이름 : 신선설농탕 고척점 도로명 주소 : 서울 구로구 경인로 387 385, 영업시간 : 매일 10:00 ~ 22:00 전화번호 : 02-2615-8080 옵션사항 : 예약, 단체 이용 가능, 주차, 포장, 남/녀 화장실 ...'}, {'url': 'https://m.blog.naver.com/zetteun/222342052693', 'content': '신선설농탕 고척점. 서울특별시 구로구 경인로 385. 운영 시간: 24시간/ 매일. 전화 번호:02-2615-8080. 존재하지 않는 이미지입니다. 고척/개봉동 맛집이라 했지만, 체인점이 워낙 많아 의미가 없다. 다른 곳은 가보진 못했지만 고척점도 굉장히 큰 편. .'}, {'url': 'https://sonsoogun.com/잡동사니/5025', 'content': '고척 신선설농탕 아이와 함께 설렁탕 먹고 온 후기. 개봉 고척 맛집 추천. ... 전화번호 : 02-2615-8080 영업시간 : 매일 10:00~22:00. ... 신선설농탕 고척점에 들어가자마자 눈에 띈 건 여기저기 전시되어 있는 예쁜 손뜨개 인형들. 우리집에도 능력자가 

In [9]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langgraph.prebuilt import ToolExecutor
from langchain_community.chat_models import ChatOllama
from langchain_core.callbacks.manager import CallbackManager
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_core.runnables import RunnableParallel


def format_search_result(search_result):
    return " ".join([r["content"] for r in search_result])


tool_executor = ToolExecutor(tools)

In [10]:
search_agent = (
    agent_executor
    | itemgetter("output")
    | ReActJsonSingleInputOutputParser()
    | tool_executor
    | format_search_result
)

search_agent_parallel = RunnableParallel(
    context=search_agent, question=itemgetter("input")
)

In [11]:
search_agent_parallel.invoke(
    {
        "input": "고척 신선설농탕 전화번호 웹 검색하여 알려줘.",
        "chat_history": [],
    }
)



> Entering new AgentExecutor chain...
Question: 고척 신선설농탕의 전화번호를 찾을 수 있는 방법은 무엇인가요?
Thought: 사용자에게 정보를 제공하기 위해 'tavily_search_results_json' 툴을 호출합니다.
Action: ```{
     "action": "tavily_search_results_json",
     "action_input": {
         "query": "고척 신선설농탕 전화번호"
     }
}```

[{'url': 'https://www.diningcode.com/profile.php?rid=tBA1JWbJxcdM', 'content': 'chazer 4월 24일. 구로구 고척동 맛집 신선설농탕 고척점 가격메뉴주차운영시간리뷰 이름 : 신선설농탕 고척점 도로명 주소 : 서울 구로구 경인로 387 385, 영업시간 : 매일 10:00 ~ 22:00 전화번호 : 02-2615-8080 옵션사항 : 예약, 단체 이용 가능, 주차, 포장, 남/녀 화장실 ...'}, {'url': 'https://m.blog.naver.com/zetteun/222342052693', 'content': '신선설농탕 고척점. 서울특별시 구로구 경인로 385. 운영 시간: 24시간/ 매일. 전화 번호:02-2615-8080. 존재하지 않는 이미지입니다. 고척/개봉동 맛집이라 했지만, 체인점이 워낙 많아 의미가 없다. 다른 곳은 가보진 못했지만 고척점도 굉장히 큰 편. .'}, {'url': 'https://sonsoogun.com/잡동사니/5025', 'content': '고척 신선설농탕 아이와 함께 설렁탕 먹고 온 후기. 개봉 고척 맛집 추천. ... 전화번호 : 02-2615-8080 영업시간 : 매일 10:00~22:00. ... 신선설농탕 고척점에 들어가자마자 눈에 띈 건 여기저기 전시되어 있는 예쁜 손뜨개 인형들. 우리집에도 능력자가 있어 손뜨개 ...'}, {'url': 'https:

ValidationError: 1 validation error for TavilyInput
query
  field required (type=value_error.missing)

In [12]:
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI Assistant. You must answer the question based on the context.\n#Context: {context}",
        ),
        ("user", "{question}"),
    ]
)

eeve = ChatOllama(
    model="EEVE-Korean-Instruct-10.8B:latest", #"EEVE-Korean-10.8B:long",
    temperature=0,
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
)

chain = (
    RunnableParallel(question=itemgetter("question"),
                     context=itemgetter("context"))
    | prompt
    | eeve
    | StrOutputParser()
)

In [13]:
final_chain = search_agent_parallel | chain

In [14]:
final_chain.invoke(
    {
        "input": "고척 신선설농탕 전화번호 검색하여 알려주세요",
        "chat_history": [],
    }
)



> Entering new AgentExecutor chain...
Question: 고척 신선설농탕의 전화번호를 검색하여 알려주세요.
Thought: 사용자로부터 고척 신선설농탕의 전화번호를 찾는 요청이 있습니다. 이에 대한 답변으로, tavily_search_results_json 도구를 이용하여 관련 정보를 검색할 수 있습니다.
Action: ```{
     "action": "tavily_search_results_json",
     "action_input": {
         "query": {
             "title": "고척 신선설농탕 전화번호",
             "description": "고척 신선설농탕의 최신 전화번호 정보를 검색하려면"
         }
     }
}```



ValidationError: 1 validation error for TavilyInput
query
  str type expected (type=type_error.str)